# Group advantages: what information survives normalization?

Day 22 · Chapter 13 · CPU worked notebook

Prerequisites: Chapter12 score-function gradients. Read Chapter13 sections13.1–13.2.

Predict: do an all-success and an all-failure group produce different relative gradients? What about rewards0,0,1,1? State your reasoning before running.

Saved reference preview from the bounded CPU reference execution. It is a fixed result; run the cells to regenerate figures after changing controls. Make the prediction above before inspecting the preview.

![Saved reference 1: group advantages](../figures/chapter-13/day-22-01_group_advantages-01.png)

In [ ]:
from pathlib import Path
import sys
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "dongxi_llms").is_dir())
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
plt.rcParams.update({"figure.figsize": (8, 3.8), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
# CPU only. Local original fixtures; no checkpoint/network/server needed.


In [ ]:
from dongxi_llms.grpo_lab import group_advantages
rewards = torch.tensor([[0., 0., 1., 1.], [1., 1., 1., 1.], [0., 0., 0., 0.]])
advantages = group_advantages(rewards)
print("Rewards [prompts,G]:", rewards)
print("Detached advantages:", advantages, "requires_grad:", advantages.requires_grad)
fig, ax = plt.subplots()
for i, row in enumerate(advantages):
    ax.plot(range(1, 5), row, marker="o", label=["Mixed", "All success", "All failure"][i])
ax.axhline(0, color="gray", linewidth=.8)
ax.set(xlabel="Response index within one prompt group", ylabel="Relative advantage")
ax.legend()
plt.show()

Both constant groups lose their relative comparison: all centered values are zero. The mixed group retains opposite directions. This is a policy-signal result; the KL term or optimizer momentum can still change parameters.

Exercise: compare population and sample standard deviation at G2 and G4. Predict which gives the larger magnitude.

In [ ]:
# Reference solution: exact scale comparison, not an optimizer-performance claim.
for values in ([0., 1.], [0., 0., 1., 1.]):
    r = torch.tensor(values, dtype=torch.float64)
    centered = r-r.mean()
    print("G =", len(r), "population =", centered/r.std(correction=0),
          "sample =", centered/r.std(correction=1))
print("Try a nearly constant continuous-reward group next; epsilon and reward scale matter.")

Interpretation: sample std is larger by sqrt(G/(G-1)); the normalized advantages shrink correspondingly. Change one reward or remove normalization, then explain which information is preserved.

Evidence boundary: these arrays verify normalization geometry, not an LLM learning gain. Worked answer: Chapter13 questions1–3.